# Title: (PalestineIsraelWar-Cleaning)

## Introduction
Dataset about all events in Palestine and Israel War 

## Process

### Import libraries

In [ ]:
%pip install ruff

In [ ]:
%pip install ydata-profiling

In [2]:
import pandas as pd  # noqa: F401
import numpy as np  # noqa: F401
import matplotlib.pyplot as plt  # noqa: F401
import seaborn as sns  # noqa: F401
from pandas_profiling import ProfileReport  # noqa: F401

C:\Users\Renter.TR-1000-004\AppData\Local\Temp\ipykernel_33080\918478641.py:5: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  from pandas_profiling import ProfileReport  # noqa: F401


### Load Data

In [3]:
df = pd.read_csv(
    "../../1_datasets/data/01_category_war_events_data/gaza_war_events/palestine_israel_conflict/data.csv"
)
df.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,ISR46155,2025-06-20,2025,1,Political violence,Explosions/Remote violence,Shelling/artillery/missile attack,Military Forces of Iran (1989-),NaN,External/Other forces,...,Haifa,32.8184,34.9885,1,BE106; Calcalist; Colbo News; Haaretz; Israel ...,Local partner-New media,"On 20 June 2025, at least one of the ballistic...",0,NaN,1750721820
1,ISR46156,2025-06-20,2025,1,Political violence,Explosions/Remote violence,Shelling/artillery/missile attack,Military Forces of Iran (1989-),NaN,External/Other forces,...,Beersheba,31.2518,34.7913,1,13 News; Calcalist; Haaretz; N12; Now 14; Time...,National,"On 20 June 2025, at least two of the ballistic...",0,NaN,1750721820
2,ISR46157,2025-06-20,2025,1,Political violence,Explosions/Remote violence,Shelling/artillery/missile attack,Military Forces of Iran (1989-),NaN,External/Other forces,...,Tel Aviv,32.0809,34.7806,3,Haaretz; Kan News; N12; Times of Israel,National,"On 20 June 2025, shrapnel from Iranian ballist...",0,NaN,1750721820
3,ISR46158,2025-06-20,2025,1,Strategic developments,Strategic developments,Disrupted weapons use,Military Forces of Israel (2022-),NaN,State forces,...,Tel Aviv,32.0809,34.7806,3,N12; Times of Israel,National,"Interception: On 20 June 2025, Israeli militar...",0,NaN,1750721820
4,ISR46159,2025-06-20,2025,1,Strategic developments,Strategic developments,Disrupted weapons use,Military Forces of Israel (2022-),NaN,State forces,...,Ne'ot HaKikar,30.9341,35.3784,2,Haaretz; Liveuamap; N12; Times of Israel; Ynet,Local partner-New media,"Interception: On 20 June 2025, Israeli militar...",0,NaN,1750721820


In [ ]:
df.columns  # You can refer to the data dictionary for more information about each column!

### Explore the dates (what period of time does it cover and how much we need?)

In [4]:
print("Oldest Event", df["event_date"].min())
print("Most recent Event", df["event_date"].max())

Oldest Event 2016-01-01
Most recent Event 2025-06-20


So basically we don't need that all events before 07/10/2023, we will drop anything before that

In [5]:
df["event_date"] = pd.to_datetime(df["event_date"])
df = df.loc[df["event_date"] >= "2023-10-07"]

In [6]:
print("Oldest Event", df["event_date"].min())
print("Most recent Event", df["event_date"].max())

Oldest Event 2023-10-07 00:00:00
Most recent Event 2025-06-20 00:00:00


We are also focusing now on the damage on Gaza, so we will drop everything any other data

In [7]:
df = df.loc[df["admin1"] == "Gaza Strip"]

### Viewing and modifing column names

In [8]:
pd.DataFrame(
    df.columns
)  # You can refer to the data dictionary for more information about each column!

,0
0,event_id_cnty
1,event_date
2,year
3,time_precision
4,disorder_type
5,event_type
6,sub_event_type
7,actor1
8,assoc_actor_1
9,inter1


As we see, no all the columns' names are descriptive

In [11]:
df = df.rename(
    columns={
        "event_id_cnty": "event_id",
        "event_date": "date",
        "time_precision": "date_precision",
        "event_type": "event_group",
        "sub_event_type": "event_subtype",
        "actor1": "primary_targeting_actor",
        "assoc_actor_1": "assoc_targeting_actor",
        "inter1": "primary_targeting_actor_type",
        "actor2": "primary_targeted_actor",
        "assoc_actor_2": "assoc_targeted_actor",
        "inter2": "primary_targeted_actor_type",
        "interaction": "interaction_type",
        "civilian_targeting": "civilian_targeted",
        "iso": "country_iso",
        "region": "region",
        "country": "country",
        "admin1": "administrative_division_1",
        "admin2": "administrative_division_2",
        "admin3": "administrative_division_3",
        "source": "sources",
        "source_scale": "source_scope",
        "fatalities": "fatality_count",
        "tags": "tags",
        "timestamp": "data_timestamp",
    }
)

### Explore Data for missing values

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20658 entries, 16 to 45559
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   event_id                      20658 non-null  object        
 1   date                          20658 non-null  datetime64[ns]
 2   year                          20658 non-null  int64         
 3   date_precision                20658 non-null  int64         
 4   disorder_type                 20658 non-null  object        
 5   event_group                   20658 non-null  object        
 6   event_subtype                 20658 non-null  object        
 7   primary_targeting_actor       20658 non-null  object        
 8   assoc_targeting_actor         569 non-null    object        
 9   primary_targeting_actor_type  20658 non-null  object        
 10  primary_targeted_actor        13201 non-null  object        
 11  assoc_targeted_actor          27

In [ ]:
pd.DataFrame(df.isnull().sum()).loc[df.isnull().sum() != 0]

### We can use the pandas profiling for this columns

In [ ]:
profile = ProfileReport(df, title="Data Quality Profiling Report")
profile

In [ ]:
profile.to_file("data_quality_report.html")

َQuick action: The region column has constant value: "Middle East" so it doesn't have any contribution to the value of our dataset, we'll drop it

In [ ]:
df.drop("region", axis=1, inplace=True)

civilian_targeted has constant value "Civilian targeting"
We will look into it on data exploration phase

### Visualize the null values

In [ ]:
null_columns = [
    "assoc_targeting_actor",
    "primary_targeted_actor",
    "assoc_targeted_actor",
    "primary_targeted_actor_type",
    "civilian_targeted",
    "administrative_division_1",
    "administrative_division_2",
    "administrative_division_3",
    "tags",
]
null_prects = df[null_columns].isnull().sum() / df.shape[0] * 100

# Plot
plt.figure(figsize=(10, 6))
null_prects.sort_values(ascending=False).plot(
    kind="bar", color="tomato", edgecolor="black"
)
plt.title("Null Values precentage per Column")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

### Handling Missing Values

1. We will drop columns with high null values precentage
2. We will replace na with Unkown value in other columns

In [ ]:
df.drop(columns=["administrative_division_3", "tags"], inplace=True)

As quick processing step We will replace the missing values with "Unkown" in the columns because we didn't explore the data enough

In [ ]:
df.fillna("Unknown", inplace=True)

In [ ]:
df.isnull().sum()

### Handling Duplicates

In [ ]:
df.duplicated().any()

### Handling Outliers

In [ ]:
# outliers in this column are actually real numbers
plt.figure(figsize=(6, 10))
df.boxplot(column="fatality_count")
plt.title("Boxplot of Fatalities")
plt.ylabel("Number of Fatalities")
plt.grid(True)
plt.show()

### Convert Data Types

In [ ]:
df.dtypes

We notice that date_precision, geo_percision is in numeric type, but it is cateogrical so we will map the values into textual values

In [ ]:
df["geo_precision_label"] = df["geo_precision"].map(
    {1: "Exact location", 2: "Admin division", 3: "Broad/Unknown area"}
)
df["date_precision_label"] = df["date_precision"].map(
    {1: "Exact date", 2: "Approximate date", 3: "Broad/Unknown date"}
)

In [ ]:
df.drop(columns=["geo_precision", "date_precision"], inplace=True)

In [ ]:
df.rename(
    columns={
        "geo_precision_label": "geo_precision",
        "date_precision_label": "date_precision",
    },
    inplace=True,
)

### Save Cleaned Data

In [ ]:
df.to_csv(
    "../../1_datasets/data/clean_datasets/palestine_israel_war_cleaned_data.csv",
    index=False,
)